In [ ]:
import pandas as pd
import numpy as np
import glob
import os
import shutil
import matplotlib.pyplot as plt

import mne
import pyprep 

from mne_bids import BIDSPath, read_raw_bids, get_entity_vals, write_raw_bids

%matplotlib widget

In [ ]:
# Define the path to the BIDS root directory
bids_root = '/media/hcs-psy-narun/Parkinson_project_chch/PD-EEG-ANON_eegOnly/sourcedata'

# Define the output path to save modified data
output_bids_root = '/media/hcs-psy-narun/Parkinson_project_chch/PD-EEG-ANON_eegOnly/sourcedataRest'

# Check if output directory exists, if not, create it
if not os.path.exists(output_bids_root):
    os.makedirs(output_bids_root)
    print(f"Created directory: {output_bids_root}")
else:
    print(f"Directory already exists: {output_bids_root}")

# Get all subjects and sessions from the BIDS dataset
subjects = get_entity_vals(bids_root, 'subject')
sessions = get_entity_vals(bids_root, 'session')
runs = get_entity_vals(bids_root, 'run')

##### Crop eeg records to 90 (start experiment) and 200 (stop experiment) markers; adjust events according to the crop

In [ ]:
# Iterate over all subjects and sessions
for subject in subjects:
    for session in sessions:
        for run in runs:
                
            # Create the BIDSPath for each subject and session
            bids_path = BIDSPath(subject=subject,
                                session=session,
                                task='restEyesClosed',
                                run=run,
                                suffix='eeg',
                                extension='.vhdr',  # EEG data in BrainVision format
                                root=bids_root,
                                datatype='eeg')

            # Check if the file exists before trying to load it
            if os.path.exists(bids_path.fpath):
                
                print(f"Processing {bids_path.fpath}")
                
                # Load the data
                raw = read_raw_bids(bids_path)

                # Extract events from annotations before cropping
                events, event_id = mne.events_from_annotations(raw)

                # Define potential event markers for start and end
                start_markers = ['90', '120']  # Pick the last one
                end_markers = ['200', '91', '121']  # Pick the first one

                # Find the last event from start markers
                tmin = None
                for marker in reversed(start_markers):
                    if marker in event_id:
                        tmin = events[events[:, 2] == event_id[marker], 0][-1] / raw.info['sfreq']
                        break

                # Find the first event from end markers
                tmax = None
                for marker in end_markers:
                    if marker in event_id:
                        tmax = events[events[:, 2] == event_id[marker], 0][0] / raw.info['sfreq']
                        break

                # Check if both start and end markers are found
                if tmin is None or tmax is None:
                    print(f"Start or end event markers not found for {bids_path.fpath}")
                    continue

                # Crop the raw data between the start and end event
                raw.crop(tmin=tmin, tmax=tmax)
                print(f"Data cropped from {tmin} to {tmax} seconds for {bids_path.fpath}")

                # Adjust event times after cropping (subtract tmin from event times)
                adjusted_events = events.copy()
                adjusted_events[:, 0] -= int(tmin * raw.info['sfreq'])  # Adjust sample numbers

                # Reverse the event_id so that the keys are the event codes and the values are descriptions
                event_desc_corrected = {v: k for k, v in event_id.items()}  # Swap keys and values

                # Re-create annotations for the cropped data based on adjusted events
                new_annotations = mne.annotations_from_events(adjusted_events, sfreq=raw.info['sfreq'], event_desc=event_desc_corrected)
                raw.set_annotations(new_annotations)

                # Save the modified raw data back to a new BIDS directory
                new_bids_path = BIDSPath(subject=subject,
                                         session=session,
                                         task='restEyesClosed',
                                         run=run,
                                         suffix='eeg',
                                         extension='.vhdr',  # EEG data in BrainVision format
                                         root=output_bids_root,
                                         datatype='eeg')

                write_raw_bids(raw, new_bids_path, overwrite=True, allow_preload=True, format='BrainVision')
                print(f"Saved cropped raw data to {new_bids_path}")
            
            else:
                # If the file does not exist, skip to the next
                continue

##### Find and mark bad channels in *channels.tsv tables

In [ ]:
# Iterate over all subjects and sessions
for subject in subjects:
    for session in sessions:
        for run in runs:
                
            # Create the BIDSPath for each subject and session
            bids_path = BIDSPath(subject=subject,
                                session=session,
                                task='restEyesClosed',
                                run=run,
                                suffix='eeg',
                                extension='.vhdr',  # EEG data in BrainVision format
                                root=output_bids_root,
                                datatype='eeg')
            #fullpath = ('/').join(str(bids_path.fpath).split('/')[:-1]) + '/eeg/'+ str(bids_path.fpath).split('/')[-1]
            # Check if the file exists before trying to load it
            if os.path.exists(bids_path.fpath):
                
                print(f"Processing {bids_path.fpath}")
                
                ## load the data
                raw = read_raw_bids(bids_path)

                ## Perform PyPREP noisy channel detection
                # filter eeg with notch filter 50hz
                raw_notch = raw.copy().load_data().notch_filter(freqs=float(raw.info.get('line_freq')))

                #Use with different parameters
                bads_detector = pyprep.NoisyChannels(raw=raw_notch , do_detrend=True, random_state=42)
                bads_detector.find_bad_by_SNR()
                bads_detector.find_bad_by_correlation(correlation_threshold=0.25)
                bads_detector.find_bad_by_deviation(deviation_threshold=8.0)
                bads_detector.find_bad_by_hfnoise(HF_zscore_threshold=8.0)
                bads_detector.find_bad_by_nan_flat()
                bads_detector.find_bad_by_ransac(n_samples=500, corr_thresh=0.6) 
                #get result
                bad_dct = bads_detector.get_bads(verbose=True, as_dict=True) #store at dct
                #print(sorted(bad_dct['bad_all']))
                #bad_dct['bad_all'].extend(['M1', 'M2', 'VEO', 'F11', 'F12', 'FT11', 'FT12'])
                
                ## Edit channel table by adding detected bad channel
                #ch table path
                tbl_ch_path = str(bids_path.fpath).split('_eeg.')[0]+'_channels.tsv'
                # #load ch table
                tbl = pd.read_csv(tbl_ch_path , sep='\t')
                tbl2 = tbl.set_index('name') #make a copy to write to
                #replace good to bad in the table
                for ch in bad_dct['bad_all']:
                    if tbl2.loc[ch,'status'] == 'bad':
                        print('already written')
                    else:
                        tbl2.loc[ch,'status'] = 'bad'
                #write bad chanels back to table
                tbl['status'] = tbl2['status'].values
                tbl['status_description'] = np.full(len(tbl['status_description']), 'n/a') #write 'n/a' as in orig, otherwise it will be empty
                #re-write the table back to folder
                tbl.to_csv(tbl_ch_path , sep='\t', index=False)
            
            
            
            
            
            else:
                #print(f"File not found: {bids_path.fpath}")
                continue

In [ ]:
import os
import pandas as pd

# Initialize lists to store electrode sets
all_electrode_sets = []

# Iterate over all subjects, sessions, and runs
for subject in subjects:
    for session in sessions:
        for run in runs:
            # Create the BIDSPath for each subject and session
            bids_path = BIDSPath(
                subject=subject,
                session=session,
                task='restEyesClosed',
                run=run,
                suffix='eeg',
                extension='.vhdr',  # EEG data in BrainVision format
                root=output_bids_root,
                datatype='eeg',
            )

            # Path to the channel table
            tbl_ch_path = str(bids_path.fpath).split('_eeg.')[0] + '_channels.tsv'

            # Check if the table exists
            if os.path.exists(tbl_ch_path):
                # Load the channel table
                tbl = pd.read_csv(tbl_ch_path, sep='\t')
                
                # Get electrode names, converted to lowercase for case-insensitive comparison
                electrode_names = set(tbl['name'].str.lower())
                all_electrode_sets.append(electrode_names)
            #else:
                #print(f"Channel table not found for {bids_path.fpath}. Skipping.")

# Determine common electrodes and dropping out electrodes
if all_electrode_sets:
    # Common electrodes (intersection of all sets)
    common_electrodes = set.intersection(*all_electrode_sets)

    # Dropping out electrodes (union of all sets minus the common ones)
    all_electrodes = set.union(*all_electrode_sets)
    dropping_out_electrodes = all_electrodes - common_electrodes

    # Print results
    print("Common electrodes across all tables:")
    print(sorted(common_electrodes))

    print("\nDropping out electrodes (not common to all tables):")
    print(sorted(dropping_out_electrodes))
else:
    print("No channel tables were found.")


In [ ]:
print(len(common_electrodes))
print(len(dropping_out_electrodes))

In [ ]:
# Iterate over all subjects and sessions
for subject in subjects:
    for session in sessions:
        for run in runs:
                
            # Create the BIDSPath for each subject and session
            bids_path = BIDSPath(subject=subject,
                                session=session,
                                task='restEyesClosed',
                                run=run,
                                suffix='eeg',
                                extension='.vhdr',  # EEG data in BrainVision format
                                root=output_bids_root,
                                datatype='eeg')
            #fullpath = ('/').join(str(bids_path.fpath).split('/')[:-1]) + '/eeg/'+ str(bids_path.fpath).split('/')[-1]
            # Check if the file exists before trying to load it
            if os.path.exists(bids_path.fpath):
                
                print(f"Processing {bids_path.fpath}")
                

                bad_dct = {}
                bad_dct['bad_all'] = ['M1', 'M2', 'VEO', 'F11', 'F12', 'FT11', 'FT12', 'FT7', 'FT8', 'PO5', 'PO6',
                                      'm1', 'm2', 'veo', 'f11', 'f12', 'ft11', 'ft12', 'ft7', 'ft8', 'po5', 'po6', 'Po6','Po5']


                ## Edit channel table by adding detected bad channel
                #ch table path
                tbl_ch_path = str(bids_path.fpath).split('_eeg.')[0]+'_channels.tsv'
                # #load ch table
                tbl = pd.read_csv(tbl_ch_path , sep='\t')
                tbl2 = tbl.set_index('name') #make a copy to write to
                #replace good to bad in the table
                for ch in bad_dct['bad_all']:
                    if ch in tbl2.index:  # Check if the channel exists in the table
                        if tbl2.loc[ch, 'status'] == 'bad':
                            print(f"Channel {ch} already marked as bad.")
                        else:
                            tbl2.loc[ch, 'status'] = 'bad'
                            print(f"Marked channel {ch} as bad.")
                    else:
                        print(f"Channel {ch} not found in the table. Skipping.")
                #write bad chanels back to table
                tbl['status'] = tbl2['status'].values
                tbl['status_description'] = np.full(len(tbl['status_description']), 'n/a') #write 'n/a' as in orig, otherwise it will be empty
                #re-write the table back to folder
                tbl.to_csv(tbl_ch_path , sep='\t', index=False)
            
            
            
            
            
            else:
                #print(f"File not found: {bids_path.fpath}")
                continue

##### Copy anat folder, rename 'defaced' files into bids-compartible

In [ ]:
# Copy anat folders for each subject and session and rename T1w files
for subject in subjects:
    for session in sessions:
        # Define the source and destination paths for the anat folder
        anat_src = os.path.join(bids_root, f'sub-{subject}', f'ses-{session}', 'anat')
        anat_dest = os.path.join(output_bids_root, f'sub-{subject}', f'ses-{session}', 'anat')

        # Check if the combination of subject-session exists and anat folder is present
        if os.path.exists(anat_src):
            # If destination folder doesn't exist, create it
            if not os.path.exists(anat_dest):
                os.makedirs(anat_dest)
            # Copy the anat folder
            shutil.copytree(anat_src, anat_dest, dirs_exist_ok=True)
            print(f"Copied anat folder for subject {subject} session {session} from {anat_src} to {anat_dest}")

            # Now rename T1-weighted MRI files that have '_defaced' in the name
            for root, dirs, files in os.walk(anat_dest):
                for file in files:
                    # Check if the file is a T1-weighted MRI file with '_defaced' in the name
                    if '_defaced' in file and file.endswith('.nii.gz'):
                        old_file_path = os.path.join(root, file)

                        # Create the new file name by removing '_defaced'
                        new_file_name = file.replace('_defaced', '')
                        new_file_path = os.path.join(root, new_file_name)

                        # Rename the file
                        print(f"Renaming: {old_file_path} -> {new_file_path}")
                        shutil.move(old_file_path, new_file_path)
        else:
            print(f"No anat folder found for subject {subject} session {session}. Skipping anat copy.")